# SciBERT Coreference Resolution Optimization

This notebook implements an optimized pipeline for Cross-Document Coreference Resolution (CDCR) of software mentions.

### Key Improvements:
1. **Supervised Contrastive Learning (SupCon)**: Better cluster density in embedding space.
2. **Metadata-Aware Heuristics**: Leverages software names, and developers to avoid false merges.
3. **Hierarchical Agglomerative Clustering (HAC)**
4. **SciBERT Encoder**: High-quality semantic representations for scientific text.

In [ ]:
!pip install transformers torch scorch scipy scikit-learn tqdm pandas pytorch-metric-learning

In [ ]:
import json
import re
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from statistics import mean
from pytorch_metric_learning import losses

try:
    from scorch import scores as scorch_scores
    SCORCH_AVAILABLE = True
except ImportError:
    SCORCH_AVAILABLE = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Software-Aware Heuristics
This section handles name canonicalization and attribute-based similarity adjustments.

In [ ]:
def canonicalize_name(name):
    if not name: return ""
    name = name.lower().strip()
    name = re.sub(r'\s+v?\.?\d+(\.\d+)*$', '', name) # Remove trailing versions

    # Common clusters
    for target in ['matlab', 'stata', 'spss', 'sas', 'beast', 'python', 'r ']:
        if target in name: return target.strip()
    return name

def extract_metadata(mention_data):
    metadata = {'version': None, 'developer': None, 'abbreviation': None}
    for rel in mention_data.get('relations', []):
        rel_type = rel.get('type', '').lower()
        m_text = rel.get('mention', '').strip()
        if 'version' in rel_type or 'release' in rel_type: metadata['version'] = m_text
        elif 'developer' in rel_type: metadata['developer'] = m_text.lower()
        elif 'abbreviation' in rel_type: metadata['abbreviation'] = m_text.lower()
    return metadata

def get_heuristic_adjustment(m1, m2):
    meta1, meta2 = extract_metadata(m1), extract_metadata(m2)
    name1, name2 = canonicalize_name(m1['mention']), canonicalize_name(m2['mention'])

    penalty = 0.0
    boost = 0.0

    # Developer Conflict
    if meta1['developer'] and meta2['developer']:
        if meta1['developer'] not in meta2['developer'] and meta2['developer'] not in meta1['developer']:
            penalty += 0.8

    # Exact Name Match
    if name1 == name2 and name1 != "":
        boost += 0.5

    return penalty, boost

## 2. Model Implementation

In [ ]:
class SiameseSciBERT(nn.Module):
    def __init__(self, projection_dim=256):
        super(SiameseSciBERT, self).__init__()
        self.bert = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')
        self.projection = nn.Sequential(
            nn.Linear(768, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )

    def forward(self, ids, mask):
        out = self.bert(ids, attention_mask=mask).last_hidden_state[:, 0, :]
        emb = self.projection(out)
        return F.normalize(emb, p=2, dim=1)

## 3. Data Loading & Training

In [ ]:
def load_data(data_path, label_path=None):
    mentions = {}
    with open(data_path, 'r') as f:
        for line in f:
            d = json.loads(line)
            mentions[d['mention_id']] = d

    id_to_label = {}
    if label_path:
        with open(label_path, 'r') as f:
            clusters = json.load(f)
            for cid, cluster in enumerate(clusters):
                for mid in cluster:
                    id_to_label[mid] = cid

    return mentions, id_to_label, list(mentions.keys())

class CorefDataset(Dataset):
    def __init__(self, mention_ids, mentions, id_to_label):
        self.ids = mention_ids
        self.mentions = mentions
        self.id_to_label = id_to_label

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        mid = self.ids[idx]
        m = self.mentions[mid]
        text = f"{m['mention']} [SEP] {m['sentence']}"
        label = self.id_to_label.get(mid, -1)
        return text, label

def collate_fn(batch, tokenizer):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=128, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels)

## 4. Evaluation Engine

In [ ]:
def evaluate_coref(gold_clusters, pred_clusters):
    if SCORCH_AVAILABLE:
        gold = [set(c) for c in gold_clusters]
        pred = [set(c) for c in pred_clusters]
        _, _, m_f = scorch_scores.muc(gold, pred)
        _, _, b_f = scorch_scores.b_cubed(gold, pred)
        _, _, c_f = scorch_scores.ceaf_e(gold, pred)
        return mean([m_f, b_f, c_f])
    else:
        # Fallback B3 implementation
        def get_m_to_c(clusters):
            m_to_c = {}
            for cid, cl in enumerate(clusters):
                for mid in cl: m_to_c[mid] = cid
            return m_to_c

        gold_m_to_c = get_m_to_c(gold_clusters)
        pred_m_to_c = get_m_to_c(pred_clusters)
        mentions = set(gold_m_to_c.keys())

        if not mentions: return 0.0
        p_sum, r_sum = 0, 0
        for m in mentions:
            g_cl = set(mm for mm in mentions if gold_m_to_c.get(mm) == gold_m_to_c.get(m))
            p_cl = set(mm for mm in mentions if pred_m_to_c.get(mm) == pred_m_to_c.get(m))
            common = len(g_cl & p_cl)
            p_sum += common / len(p_cl)
            r_sum += common / len(g_cl)

        p, r = p_sum / len(mentions), r_sum / len(mentions)
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

## 5. Optimized Clustering (HAC + Heuristics)

In [ ]:
def cluster_with_hac(embeddings, mention_ids, mentions, threshold=0.5):
    N = len(mention_ids)
    sims = np.dot(embeddings, embeddings.T)
    dists = 1.0 - sims
    np.fill_diagonal(dists, 0)

    for i in range(N):
        for j in range(i+1, N):
            p, b = get_heuristic_adjustment(mentions[mention_ids[i]], mentions[mention_ids[j]])
            adj = dists[i, j] + p - b
            dists[i, j] = dists[j, i] = max(0.01, min(1.0, adj))

            # Strict software separation
            n1 = canonicalize_name(mentions[mention_ids[i]]['mention'])
            n2 = canonicalize_name(mentions[mention_ids[j]]['mention'])
            if n1 != n2 and n1 != "" and n2 != "":
                dists[i, j] = dists[j, i] = max(dists[i, j], 0.8)

    Z = linkage(squareform(dists, checks=False), method='average')
    labels = fcluster(Z, t=threshold, criterion='distance')

    res = {}
    for i, l in enumerate(labels):
        res.setdefault(l, []).append(mention_ids[i])
    return list(res.values())


## 6. Execution Loop (with Early Stopping)
Runs for up to 30 epochs with patience of 4 epochs for early stopping.

In [ ]:
# SET PATHS HERE
TRAIN_DATA = 'subtask1_dataset/train_data.jsonl'
TRAIN_LABELS = 'subtask1_dataset/train_labels.json'
TEST_DATA = 'subtask1_dataset/test_data.jsonl'
tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
mentions, id_to_label, m_ids = load_data(TRAIN_DATA, TRAIN_LABELS)
import random
# Load clusters for cluster-level splitting
with open(TRAIN_LABELS, 'r') as f:
    all_gold = json.load(f)
# Cluster-level split (80/20)
random.seed(42)
shuffled_clusters = all_gold[:]
random.shuffle(shuffled_clusters)
split_idx = int(len(shuffled_clusters) * 0.8)
train_clusters = shuffled_clusters[:split_idx]
val_gold = shuffled_clusters[split_idx:]
train_ids = [m for c in train_clusters for m in c]
val_ids = [m for c in val_gold for m in c]
print(f"Split Summary: {len(train_clusters)} train clusters ({len(train_ids)} mentions), "
      f"{len(val_gold)} val clusters ({len(val_ids)} mentions)")
train_ds = CorefDataset(train_ids, mentions, id_to_label)
val_ds = CorefDataset(val_ids, mentions, id_to_label)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=lambda b: collate_fn(b, tokenizer))
model = SiameseSciBERT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = losses.SupConLoss()
# Early Stopping Setup
best_val_f1 = -1.0
patience = 4
epochs_no_improve = 0
max_epochs = 30
best_model_path = 'best_model.pth'
for epoch in range(max_epochs):
    print(f"\n--- Epoch {epoch+1}/{max_epochs} ---")
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Training")
    for ids, mask, labels in pbar:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        feats = model(ids, mask)
        loss = criterion(feats, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    avg_train_loss = train_loss / len(train_loader)
    # Inference & Clustering for Validation F1
    model.eval()
    all_embs = []
    with torch.no_grad():
        val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,
                               collate_fn=lambda b: collate_fn(b, tokenizer))
        for ids, mask, _ in tqdm(val_loader, desc="Evaluation", leave=False):
            ids, mask = ids.to(device), mask.to(device)
            all_embs.append(model(ids, mask).cpu().numpy())
    val_embeddings = np.vstack(all_embs)
    pred_clusters = cluster_with_hac(val_embeddings, val_ids, mentions, threshold=0.55)
    val_f1 = evaluate_coref(val_gold, pred_clusters)
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val CoNLL F1 = {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved with F1: {val_f1:.4f}")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs.")
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break
print(f"\nTraining complete. Best Validation CoNLL F1: {best_val_f1:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Split Summary: 586 train clusters (2431 mentions), 147 val clusters (543 mentions)


pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

...

Evaluation:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 9: Train Loss = 0.1670, Val CoNLL F1 = 0.9436
No improvement for 3 epochs.

--- Epoch 10/30 ---


Training:   0%|          | 0/152 [00:00<?, ?it/s]

Evaluation:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 10: Train Loss = 0.1914, Val CoNLL F1 = 0.9436
No improvement for 4 epochs.
Early stopping triggered after 10 epochs.

Training complete. Best Validation CoNLL F1: 0.9457


## 7. Final Inference & clusters.json Generation
This cell loads the best model and generates coreference clusters for the test data.

In [ ]:
print("\n" + "=" * 80)
print("FINAL INFERENCE ON TEST DATA")
print("=" * 80)

# Load test data
test_mentions, _, test_ids = load_data(TEST_DATA)
test_ds = CorefDataset(test_ids, test_mentions, {}) # No labels for test

# Load best model
model.load_state_dict(torch.load(best_model_path))
model.eval()

# Generate embeddings
test_embs = []
with torch.no_grad():
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                           collate_fn=lambda b: collate_fn(b, tokenizer))
    for ids, mask, _ in tqdm(test_loader, desc="Testing"):
        ids, mask = ids.to(device), mask.to(device)
        test_embs.append(model(ids, mask).cpu().numpy())

test_embeddings = np.vstack(test_embs)

# Cluster with optimized HAC
print("Clustering test mentions...")
test_clusters = cluster_with_hac(test_embeddings, test_ids, test_mentions, threshold=0.55)

# Save to clusters.json
with open('clusters.json', 'w') as f:
    json.dump(test_clusters, f, indent=2)

print(f"\nSaved {len(test_clusters)} clusters to clusters.json")
print("=" * 80)


FINAL INFERENCE ON TEST DATA


Testing:   0%|          | 0/24 [00:00<?, ?it/s]

Clustering test mentions...

Saved 259 clusters to clusters.json
